In [203]:
import numpy as np
import pandas as pd

from app.data_loader import DataLoader

In [204]:
from sentence_transformers import SentenceTransformer, util

In [205]:
embedders = [
    "cointegrated/rubert-tiny2",
    "sergeyzh/rubert-tiny-turbo",
    "sergeyzh/rubert-mini-sts",
    "deepvk/USER-base"
]

In [206]:
model = SentenceTransformer(embedders[3])

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

In [207]:
data = DataLoader("../data/generated_questions").test_cases

In [208]:
def cosine_similarity(a, b):
    return float(util.cos_sim(a, b)[0][0])

In [209]:
scores = pd.DataFrame()

for test_case_uuid in data:
    test_case = data[test_case_uuid]

    ref_embedding = model.encode("passage:" + test_case.question.reference_answer)

    for student_answer in test_case.answers:
        student_embedding = model.encode("query:" + student_answer.text)
        similarity = cosine_similarity(ref_embedding, student_embedding)

        new_row = pd.DataFrame(
            [
                {
                    "answer_type": student_answer.answer_type,
                    "real_score": student_answer.quality,
                    "cosine_similarity": similarity,
                    # "ref_answ": test_case.question.reference_answer,
                    # "stud_answ": student_answer.text
                }
            ]
        )
        scores = pd.concat([scores, new_row], ignore_index=True)

In [210]:
scores["real_score"].corr(scores["cosine_similarity"])

np.float64(0.8232680042751798)

In [211]:
for answer_type in scores["answer_type"].unique():
    df_slice = scores[scores["answer_type"] == answer_type]

    print(answer_type)
    print(f"Similarity Mean: {df_slice["cosine_similarity"].mean()}")
    print(f"Similarity Std: {df_slice["cosine_similarity"].std()}")
    print(f"Corr real_score-similarity: {df_slice["real_score"].corr(scores["cosine_similarity"], method="spearman")}")

Отличный
Similarity Mean: 0.8169803535938263
Similarity Std: 0.05121521278358632
Corr real_score-similarity: 0.09064185246564269
Хороший
Similarity Mean: 0.7492634081840515
Similarity Std: 0.07334327798102842
Corr real_score-similarity: -0.006249963173160338
Средний
Similarity Mean: 0.6681746050715447
Similarity Std: 0.08589298756028514
Corr real_score-similarity: 0.0064527558295964715
Слабый
Similarity Mean: 0.6041066735982895
Similarity Std: 0.08400311493435297
Corr real_score-similarity: 0.16169832040458504
Очень слабый
Similarity Mean: 0.42512892320752144
Similarity Std: 0.13014859705374887
Corr real_score-similarity: 0.33011948999874846
